In [2]:
import os, json
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_dir = 'rice_leaf_diseases'  # folder under /notebooks
print("Classes found:", os.listdir(base_dir))


Classes found: ['Bacterial leaf blight', 'Brown spot', 'Leaf smut']


In [3]:
img_size = (224, 224)
batch_size = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2,   # 80% train / 20% val
)

train_gen = datagen.flow_from_directory(
    base_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    base_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

num_classes = train_gen.num_classes
class_indices = train_gen.class_indices
print("Class indices:", class_indices)

with open('rice_class_indices.json', 'w') as f:
    json.dump(class_indices, f, indent=2)


Found 96 images belonging to 3 classes.
Found 24 images belonging to 3 classes.
Class indices: {'Bacterial leaf blight': 0, 'Brown spot': 1, 'Leaf smut': 2}


In [4]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # freeze base to start

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)
model = models.Model(inputs, outputs)

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_gen,
    epochs=10,
    validation_data=val_gen
)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 16s 4s/step - accuracy: 0.3646 - loss: 1.5139 - val_accuracy: 0.5000 - val_loss: 1.0471
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - accuracy: 0.2396 - loss: 1.3704 - val_accuracy: 0.5417 - val_loss: 0.9668
Epoch 3/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - accuracy: 0.4271 - loss: 1.1473 - val_accuracy: 0.6667 - val_loss: 0.8276
Epoch 4/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 6s 2s/step - accuracy: 0.6146 - loss: 0.8829 - val_accuracy: 0.7917 - val_loss: 0.6019
Epoch 5/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - accuracy: 0.6458 - loss: 0.8544 - val_accuracy: 0.9167 - val_loss: 0.5572
Epoch 6/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - accuracy: 0.5938 - loss: 0.8430 - val_accuracy: 0.9167 - val_loss: 0.5052
Epoch 7/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step - accuracy: 0.7083 - loss: 0.7481 - val_accuracy: 0.8333 - val_loss: 0.4838
Epoch 8/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step - accuracy: 0.7292 - loss: 0.6674 - val_ac

In [5]:
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_ft = model.fit(
    train_gen,
    epochs=5,
    validation_data=val_gen
)


Epoch 1/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 15s 3s/step - accuracy: 0.6354 - loss: 0.8345 - val_accuracy: 0.9583 - val_loss: 0.3097
Epoch 2/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step - accuracy: 0.6771 - loss: 0.7635 - val_accuracy: 0.8750 - val_loss: 0.4372
Epoch 3/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step - accuracy: 0.6979 - loss: 0.7166 - val_accuracy: 0.9167 - val_loss: 0.3317
Epoch 4/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - accuracy: 0.6667 - loss: 0.6533 - val_accuracy: 0.9583 - val_loss: 0.3506
Epoch 5/5
3/3 ━━━━━━━━━━━━━━━━━━━━ 4s 2s/step - accuracy: 0.6979 - loss: 0.6687 - val_accuracy: 0.9583 - val_loss: 0.3239


In [6]:
model.save('rice_disease_cnn_model.keras')
print("Saved model as rice_disease_cnn_model.keras")



Saved model as rice_disease_cnn_model.keras
